In [ ]:
#!git clone https://github.com/whyhardt/SPICE.git

In [ ]:
# !pip install -e SPICE

In [1]:
import sys
import torch

from spice import SpiceEstimator, SpiceConfig, BaseModel

sys.path.append('../../..')
from weinhardt2026.utils.benchmarking_gru import GRUModel, training
from benchmarking_ganesh2024a import get_dataset, generate_behavior, BayesianModel

# Load dataset

Let's load the data first with the `csv_to_dataset` method. This method returns a `SpiceDataset` object which we can use right away 

In [2]:
path_data = 'data/ganesh2024a_choice.csv'
test_sessions = (3, 6, 9)
dataset_train, dataset_test, info_dataset = get_dataset(path_data=path_data, test_sessions=test_sessions, verbose=True)

Shape of dataset: torch.Size([1176, 24, 1, 11])
Number of participants: 98
Number of actions: 2


# SPICE Setup

Now we are going to define the configuration for SPICE with a `SpiceConfig` object.

The `SpiceConfig` takes as arguments 
1. `library_setup (dict)`: Defining the variable names of each module.
2. `memory_state (dict)`: Defining the memory state variables and their initial values.
3. `states_in_logit (list)`: Defining which of the memory state variables are used later for the logit computation. This is necessary for some background processes.  

And now we are going to define the SPICE model which is a child of the `BaseModel` and `torch.nn.Module` class and takes as required arguments:
1. `spice_config (SpiceConfig)`: previously defined SpiceConfig object
2. `n_actions (int)`: number of possible actions in your dataset (including non-displayed ones if applicable).
3. `n_participants (int)`: number of participants in your dataset.

As usual for a `torch.nn.Module` we have to define at least the `__init__` method and the `forward` method.
The `forward` method gets called when computing a forward pass through the model and takes as inputs `(inputs (SpiceDataset.xs), prev_state (dict, default: None), batch_first (bool, default: False))` and returns `(logits (torch.Tensor, shape: (n_participants*n_blocks*n_experiments, timesteps, n_actions)), updated_state (dict))`. Two necessary method calls inside the forward pass are:
1. `self.init_forward_pass(inputs, prev_state) -> SpiceSignals`: returns a `SpiceSignals` object which carries all relevant information already processed.
2. `self.post_forward_pass(SpiceSignals, batch_first) -> SpiceSignals`: does some re-arranging of the logits to adhere to `batch_first`.

In [3]:
spice_config = SpiceConfig(
    library_setup={
        'perception_certainty': ['contr_diff[t]'],
        'reward_learning_chosen': ['reward[t]', 'certainty[t]'],
        'reward_learning_unchosen': ['certainty[t]'],
        'choice_persistance': ['choice[t]', 'certainty[t]', 'certainty_next[t+1]'],
    },
    
    memory_state={
        'value_reward_contrast': 0.,
        'value_choice_contrast': 0.,
    },
)

In [4]:
class SPICERNN(BaseModel):
    
    def __init__(self, deterministic_perception=False, **kwargs):
        super().__init__(**kwargs)
        
        self.dropout = 0.1
        self.deterministic_perception = deterministic_perception
        
        self.participant_embedding = self.setup_embedding(num_embeddings=self.n_participants, embedding_size=self.embedding_size, dropout=self.dropout)
        
        # perception: signed contr_diff → sigmoid
        self.setup_module(key_module='perception_certainty', input_size=1+self.embedding_size, include_state=False, dropout=self.dropout)
        # chosen item: learns from reward, modulated by certainty
        self.setup_module(key_module='reward_learning_chosen', input_size=2+self.embedding_size, dropout=self.dropout)
        # unchosen item: forgetting/persistence dynamics, modulated by certainty
        self.setup_module(key_module='reward_learning_unchosen', input_size=1+self.embedding_size, dropout=self.dropout)
        # choice persistance module
        self.setup_module(key_module='choice_persistance', input_size=3+self.embedding_size, dropout=self.dropout)
        
    def forward(self, inputs, state = None):
        spice_signals = self.init_forward_pass(inputs, state)
        
        # feature extraction
        # scalar contrast difference per trial (T, W, E, B)
        cd_current = spice_signals.additional_inputs[..., 0]
        cd_next = spice_signals.additional_inputs[..., 1].unsqueeze(-1)
        # repeated to n_actions for module inputs (T, W, E, B, n_actions)
        contr_diff_current = cd_current.unsqueeze(-1).repeat(1, 1, 1, 1, self.n_actions)
        contr_diff_next = cd_next.repeat(1, 1, 1, 1, self.n_actions)
        
        # Map actions: position space (left=0, right=1) → item space (low=0, high=1)
        # cd <= 0: left=low, right=high;  cd > 0: left=high, right=low
        chose = spice_signals.actions.argmax(dim=-1)  # (T, W, E, B)
        action_contrast = torch.zeros_like(spice_signals.actions)
        # chose_low: chose left when left=low (cd<=0), OR chose right when right=low (cd>0)
        action_contrast[..., 0] = (((cd_current <= 0) & (chose == 0)) | ((cd_current > 0) & (chose == 1))).float()
        # chose_high: chose right when right=high (cd<=0), OR chose left when left=high (cd>0)
        action_contrast[..., 1] = (((cd_current <= 0) & (chose == 1)) | ((cd_current > 0) & (chose == 0))).float()
        
        # Scalar reward per trial (sum over one-hot reward vector)
        reward_scalar = spice_signals.rewards.sum(dim=-1, keepdim=True).expand_as(spice_signals.actions)
        
        participant_embeddings = self.participant_embedding(spice_signals.participant_ids)
        
        for trial in spice_signals.trials:
            
            # --- Perception: p(left=low) for current trial ---
            certainty_current = self.call_module(
                key_module='perception_certainty',
                inputs=contr_diff_current[trial],#.abs(),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings,
                activation_rnn=torch.nn.functional.sigmoid,
                # activation_rnn=torch.nn.functional.tanh,
            )
            
            certainty_next = self.call_module(
                key_module='perception_certainty',
                inputs=contr_diff_next[trial],#.abs(),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings,
                activation_rnn=torch.nn.functional.sigmoid,
                # activation_rnn=torch.nn.functional.tanh,
            )

            # --- Reward learning with explicit chosen/unchosen split ---
            # Hard masks use the deterministic item space action (ground truth mapping).
            # Certainty is passed as input so the GRU can learn to attenuate updates
            # when the deterministic assignment is unreliable.
            
            # Chosen item: update value based on actual reward
            self.call_module(
                key_module='reward_learning_chosen',
                key_state='value_reward_contrast',
                action_mask=action_contrast[trial],
                inputs=(
                    reward_scalar[trial], 
                    certainty_current,
                    ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings,
            )
            
            # Unchosen item: can learn forgetting, persistence, or drift
            self.call_module(
                key_module='reward_learning_unchosen',
                key_state='value_reward_contrast',
                action_mask=1 - action_contrast[trial],
                inputs=(certainty_current,),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings,
            )
            
            # Choice persistance
            self.call_module(
                key_module='choice_persistance',
                key_state='value_choice_contrast',
                inputs=(
                    action_contrast[trial],
                    certainty_current,
                    certainty_next,
                    ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings,
            )
            
            # --- Decision: soft-map item values to action space ---
                        
            logits_contrast = self.state['value_reward_contrast'] + self.state['value_choice_contrast']
            
            # deriving p(left=low) from contrast-difference-based certainty to assign learned reward values to given options; range=(0.5, 1.0)
            # TODO: add sign information about contrast difference
            # certainty_next = (certainty_next + 1) / 2
            certainty_next = certainty_next / 2 + 0.5
            mixed_logits_item_space = logits_contrast * certainty_next + logits_contrast.flip(-1) * (1 - certainty_next)
            
            # map mixed logits from item space (low, high) into action space (left, right)
            spice_signals.logits[trial] = torch.where(cd_next[trial] < 0, mixed_logits_item_space, mixed_logits_item_space.flip(-1))
            
        spice_signals = self.post_forward_pass(spice_signals)
        
        return spice_signals.logits, self.get_state()

Let's setup now the `SpiceEstimator` object and fit it to the data!

We are going to fit SPICE once without SINDy as a regularizer for the RNN (`sindy_weight=0`) to get peak SPICE performance given the architecture.

Then we are fitting SPICE in the normal way to get the interpretable model.

In [ ]:
estimator = SpiceEstimator(
    # model paramaeters
    spice_class=SPICERNN,
    spice_config=spice_config,
    n_actions=info_dataset['n_actions'],
    n_participants=info_dataset['n_participants'],
    
    epochs=100,
    sindy_weight=0,
    sindy_alpha=0.0001,
    sindy_pruning_frequency=50,
    sindy_threshold_pruning=0.05,
    sindy_ensemble_pruning=0.01,
    sindy_library_polynomial_degree=2,
    
    verbose=True,
)

print(f"\nStarting training on {estimator.device}...")
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)
print("\nTraining complete!")

In [5]:
path_spice = 'params/spice_ganesh2024a.pkl'

estimator = SpiceEstimator(
    # model paramaeters
    spice_class=SPICERNN,
    spice_config=spice_config,
    n_actions=info_dataset['n_actions'],
    n_participants=info_dataset['n_participants'],
    
    epochs=1000,
    warmup_steps=100,
    learning_rate=0.01,
    batch_size=None,
    ensemble_size=10,

    sindy_weight=0.1,
    sindy_alpha=0.0001,
    sindy_pruning_frequency=50,
    sindy_threshold_pruning=0.05,
    sindy_ensemble_pruning=0.01,
    sindy_library_polynomial_degree=2,
    
    verbose=True,
    save_path_spice=path_spice,
)


In [ ]:
# fitting SPICE with SINDy as RNN-regularizer
print(f"\nStarting training on {estimator.device}...")
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

Once a model is trained and saved we can load it from the pickle file using `SpiceEstimator.load_spice(path_to_model)`

In [6]:
estimator.load_spice(path_spice)

The SPICE models can be printed to the console using `SpiceEstimator.print_spice_model(participant_id)` 

In [7]:
# Print example SPICE model
participant_id = 0
print(f"\nExample SPICE model for participant {participant_id}:")
estimator.print_spice_model(participant_id=participant_id)


Example SPICE model for participant 0:
perception_certainty[t+1]     = -0.671 1 + 0.037 contr_diff[t] + 0.226 contr_diff[t]^2 
reward_learning_chosen[t+1]   = 0.208 1 + 0.933 reward_learning_chosen[t] + 0.552 reward[t] + 0.034 certainty[t] + 0.021 reward_learning_chosen^2 + -0.293 reward_learning_chosen*reward[t] + 0.296 reward_learning_chosen*certainty[t] + 0.552 reward[t]^2 + -1.276 reward[t]*certainty[t] + -0.362 certainty[t]^2 
reward_learning_unchosen[t+1] = 0.0 1 + 0.713 reward_learning_unchosen[t] + -0.045 certainty[t] + 0.039 reward_learning_unchosen^2 + 0.026 reward_learning_unchosen*certainty[t] + 0.001 certainty[t]^2 
choice_persistance[t+1]       = 0.278 1 + 1.064 choice_persistance[t] + 0.04 choice[t] + -0.148 certainty[t] + -0.156 certainty_next[t+1] + -0.002 choice_persistance^2 + -0.039 choice_persistance*choice[t] + 0.052 choice_persistance*certainty[t] + 0.059 choice_persistance*certainty_next[t+1] + 0.039 choice[t]^2 + 0.0 choice[t]*certainty[t] + 0.003 choice[t]*ce

# Benchmark models

## Classic benchmark model 

In [8]:
bay = BayesianModel(n_participants=info_dataset['n_participants'], batch_first=True)
path_bay = path_spice.replace('spice_', 'bay_')

In [ ]:
epochs = 1000
optimizer = torch.optim.Adam(bay.parameters(), lr=0.01)

gru = training(
    model=bay,
    optimizer=optimizer,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    device=torch.device('cpu'),
    )

torch.save(gru.state_dict(), path_bay)
print("Trained benchmark parameters saved to " + path_bay)

In [9]:
bay.load_state_dict(torch.load(path_bay))

<All keys matched successfully>

## GRU benchmark model

In [25]:
gru = GRUModel(
    n_actions=info_dataset['n_actions'], 
    n_participants=info_dataset['n_participants'],
    additional_inputs=2, 
    dropout=0.25,
    embedding_size=8,
    hidden_size=8,
    )
path_gru = path_spice.replace('spice_', 'gru_')

In [26]:
epochs = 1000
optimizer = torch.optim.Adam(gru.parameters(), lr=0.01)

gru = training(
    model=gru,
    optimizer=optimizer,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    device=torch.device('cpu'),
    )

torch.save(gru.state_dict(), path_gru)
print("Trained GRU parameters saved to " + path_gru)

Epoch 1/1000: L(Train): 0.6984047293663025; L(Test): 0.6940329074859619
Epoch 2/1000: L(Train): 0.694598376750946; L(Test): 0.6933453679084778
Epoch 3/1000: L(Train): 0.69452303647995; L(Test): 0.6931271553039551
Epoch 4/1000: L(Train): 0.6935860514640808; L(Test): 0.6931695342063904
Epoch 5/1000: L(Train): 0.6934753656387329; L(Test): 0.6931654810905457
Epoch 6/1000: L(Train): 0.6931096911430359; L(Test): 0.6931254267692566
Epoch 7/1000: L(Train): 0.6930813193321228; L(Test): 0.6931303143501282
Epoch 8/1000: L(Train): 0.6932721734046936; L(Test): 0.6931711435317993
Epoch 9/1000: L(Train): 0.6929399967193604; L(Test): 0.6932154893875122
Epoch 10/1000: L(Train): 0.692875862121582; L(Test): 0.6932389736175537
Epoch 11/1000: L(Train): 0.6926403045654297; L(Test): 0.6932514905929565
Epoch 12/1000: L(Train): 0.6924624443054199; L(Test): 0.6932701468467712
Epoch 13/1000: L(Train): 0.6921969652175903; L(Test): 0.6932907700538635
Epoch 14/1000: L(Train): 0.6924300789833069; L(Test): 0.69329226

In [11]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))

<All keys matched successfully>

# Analysis

In [27]:
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions

## Analysis of trial averaged accuracy

In [28]:
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    gru_model=gru,
    benchmark_model=bay,
    )

Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


,Trial Lik.,(std),n_parameters,(std),NLL,AIC,BIC
Benchmark,0.612598,0.137542,2.0,0.0,3457.769287,6919.538574,6933.261719
GRU,0.635248,0.142468,1362.0,0.0,3201.591309,9127.182617,18472.726562
SPICE-RNN,0.500161,0.005070,6176.0,0.0,4888.571289,22129.142578,64506.585938
SPICE,0.497031,0.012266,34.0,0.0,4932.875000,9933.750000,10167.045898


## Analysis of individual differences (only SPICE)

This function analyses individual differences of the SPICE models w.r.t. the fitted SINDy coefficients.

Here you can choose between a discrete odds ratio analysis between groups (e.g. `HighAccumulatedReward` vs `LowAccumulatedReward`) or a beta-effect analysis of a continuous variable (e.g. `AccumulatedReward`).

The analyses check whether the given criterion is a good predictor of the SINDy coefficients in the model.

In [ ]:
analysis_coefficients_individuals(
    path_model=path_spice,
    path_data=path_data,
    criterion="SomePerformanceColumnInDataset",
    analysis="disc", # also: "cont"
    reference="ReferenceGroupFromPerformanceColumn", # only necessary if analysis="disc"
    dir_output="../data/ganesh2024a/",
    model_class=SPICERNN,
    model_config=spice_config,
)